In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

print("=" * 70)
print("SEARCHING MODEL ARTIFACTS")
print("=" * 70)

print("Project root:")
print(PROJECT_ROOT)

print("\nModel / Joblib files:")

model_files = []

for pattern in [
    "*.pkl",
    "*.joblib",
    "*.pickle",
]:
    model_files.extend(
        PROJECT_ROOT.rglob(pattern)
    )

model_files = sorted(
    set(model_files)
)

if not model_files:
    print("❌ No model files found.")
else:
    for path in model_files:
        try:
            size_mb = path.stat().st_size / (1024 ** 2)
        except:
            size_mb = 0

        print(
            f"{size_mb:8.2f} MB | {path}"
        )

print("\n" + "=" * 70)
print("SEARCH FINISHED")
print("=" * 70)

SEARCHING MODEL ARTIFACTS
Project root:
c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction

Model / Joblib files:
  311.07 MB | c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\prediction\models\extra_trees_spatial.joblib
    1.09 MB | c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\backend\prediction\models\hgb_spatial.joblib
    0.65 MB | c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\dataset\METR-LA\adj_METR-LA.pkl
    0.65 MB | c:\Users\alisaleh\Desktop\EVERYTHING\python projects\AIO\Innoverse_world\Smart Urban Mobility Patterns Prediction\dataset\METR-LA\adj_mx_METR-LA.pkl

SEARCH FINISHED


In [2]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

et_path = (
    PROJECT_ROOT
    / "backend"
    / "prediction"
    / "models"
    / "extra_trees_spatial.joblib"
)

hgb_path = (
    PROJECT_ROOT
    / "backend"
    / "prediction"
    / "models"
    / "hgb_spatial.joblib"
)

print("=" * 70)
print("LOADING FINAL TRAFFIC MODELS")
print("=" * 70)

print("\nLoading Extra Trees...")
et_model = joblib.load(et_path)

print("Loading HGB...")
hgb_model = joblib.load(hgb_path)

print("\n" + "=" * 70)
print("MODEL TYPES")
print("=" * 70)

print("Extra Trees:")
print(type(et_model))

print("\nHGB:")
print(type(hgb_model))

print("\n" + "=" * 70)
print("EXTRA TREES ATTRIBUTES")
print("=" * 70)

print("n_features_in_:",
      getattr(et_model, "n_features_in_", None))

print("feature_names_in_:",
      getattr(et_model, "feature_names_in_", None))

print("\n" + "=" * 70)
print("HGB ATTRIBUTES")
print("=" * 70)

print("n_features_in_:",
      getattr(hgb_model, "n_features_in_", None))

print("feature_names_in_:",
      getattr(hgb_model, "feature_names_in_", None))

LOADING FINAL TRAFFIC MODELS

Loading Extra Trees...
Loading HGB...

MODEL TYPES
Extra Trees:
<class 'sklearn.ensemble._forest.ExtraTreesRegressor'>

HGB:
<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingRegressor'>

EXTRA TREES ATTRIBUTES
n_features_in_: 28
feature_names_in_: None

HGB ATTRIBUTES
n_features_in_: 28
feature_names_in_: None


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

keywords = [
    "neighbor_mean",
    "rolling_mean_3",
    "speed_change_3",
    "temp_humidity_interaction",
    "speed_vs_mean_6",
]

print("=" * 70)
print("SEARCHING FINAL FEATURE BUILDER")
print("=" * 70)

found = set()

for py_file in PROJECT_ROOT.rglob("*.py"):

    try:
        text = py_file.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matches = [
        k for k in keywords
        if k in text
    ]

    if matches:

        found.add(py_file)

        print("\nFILE:")
        print(py_file)

        print("MATCHES:")
        for m in matches:
            print("  -", m)

print("\n" + "=" * 70)
print("FILES FOUND:", len(found))
print("=" * 70)

SEARCHING FINAL FEATURE BUILDER

FILES FOUND: 0


In [ ]:

from datetime import datetime
from typing import Any


class Context_Engine:

    def __init__(
        self,
        routing_api,
        weather_api,
        holiday_api,
        event_api=None,
        accident_engine=None,
        traffic_engine=None,
    ):

        self.routing_api = routing_api
        self.weather_api = weather_api
        self.holiday_api = holiday_api
        self.event_api = event_api

        self.accident_engine = accident_engine
        self.traffic_engine = traffic_engine


   
    def build_context(
        self,
        origin,
        destination,
        departure_date,
        departure_time,
        transport_mode="car",
        route_preference="fastest",
    ):

        departure_datetime = datetime.fromisoformat(
            f"{departure_date}T{departure_time}"
        )

       
        route = self.routing_api.get_route(
            origin,
            destination
        )

        if route is None:

            raise RuntimeError(
                "Routing API failed."
            )


       
        weather = self.weather_api.get_weather(
            latitude=origin[0],
            longitude=origin[1],
            target_datetime=departure_datetime
        )

        if weather is None:

            raise RuntimeError(
                "Weather API failed."
            )


        
        holiday = self.holiday_api.is_holiday(
            departure_date,
            country="US"
        )


      
        events = None

        if self.event_api is not None:

            try:

                events = self.event_api.get_events(
                    origin=origin,
                    destination=destination,
                    departure_datetime=departure_datetime
                )

            except Exception as error:

                print(
                    f"[ContextEngine] Event API warning: {error}"
                )


        accident = None

        if self.accident_engine is not None:

            try:

                accident = (
                    self.accident_engine.predict(
                        route=route,
                        departure_datetime=departure_datetime
                    )
                )

            except Exception as error:

                print(
                    f"[ContextEngine] Accident engine warning: {error}"
                )


     
        context = {

            "request": {

                "origin": origin,

                "destination": destination,

                "departure_date":
                    departure_date,

                "departure_time":
                    departure_time,

                "departure_datetime":
                    departure_datetime.isoformat(),

                "transport_mode":
                    transport_mode,

                "route_preference":
                    route_preference,
            },


            "route":
                route,


            "weather":
                weather,


            "holiday":
                holiday,


            "events":
                events,


            "accident":
                accident,
        }


        return context